# One-Class SVM

A One-Class Support Vector Machine learns a boundary around clean training data. New points outside the boundary are novel. The kernel lets the boundary follow curved or clustered shapes.

In [ ]:
Pollis.packages("LIBSVM", "Plots")

In [ ]:
using LIBSVM, Plots, Statistics, Random

Random.seed!(1);

## Training and Test Data

The normal data forms two clusters. The test set holds new normal points plus a few that fall between or outside the clusters.

In [ ]:
cluster(c, n) = c' .+ 0.6 .* randn(n, 2)
X_train = vcat(cluster([-2.0, 0.0], 150), cluster([2.0, 1.0], 150))
X_test  = vcat(cluster([-2.0, 0.0], 20), cluster([2.0, 1.0], 20), [0.0 0.5; 0.0 -2.5; 4.5 4.0; -5.0 2.0])
is_true = vcat(falses(40), trues(4))

scatter(X_train[:, 1], X_train[:, 2]; color = :lightgray, markerstrokewidth = 0, label = "Training")
scatter!(X_test[:, 1], X_test[:, 2]; group = ifelse.(is_true, "Test: new kind", "Test: normal"), markerstrokewidth = 0)

## Fit the Model

Standardise with the **training** mean and standard deviation, and apply the same transformation to the test data. LIBSVM expects observations in columns.

In [ ]:
mu, sd = mean(X_train, dims = 1), std(X_train, dims = 1)
Ztrain = permutedims((X_train .- mu) ./ sd)
Ztest  = permutedims((X_test .- mu) ./ sd)

model = svmtrain(Ztrain; svmtype = OneClassSVM, nu = 0.05)
inside, dec = svmpredict(model, Ztest)
scores = -dec[1, :]   # higher = further outside
is_novel = .!inside

println("Novel test points: ", findall(is_novel))
println("New-kind points caught: ", sum(is_novel .& is_true), " of ", sum(is_true))

## The Boundary

Evaluate the decision function on a grid to see the boundary. It wraps both clusters and leaves the gap between them outside.

In [ ]:
xs, ys = range(-6, 6; length = 120), range(-4, 5; length = 100)
grid = permutedims(([repeat(xs, outer = length(ys)) repeat(ys, inner = length(xs))] .- mu) ./ sd)
_, gdec = svmpredict(model, grid)

contour(xs, ys, reshape(gdec[1, :], length(xs), length(ys))'; levels = [0.0], color = :black, linewidth = 2, colorbar = false)
scatter!(X_train[:, 1], X_train[:, 2]; color = :lightgray, markerstrokewidth = 0, label = "Training")
scatter!(X_test[is_novel, 1], X_test[is_novel, 2]; color = :red, marker = :star5, label = "Novel")

## nu Is the False-Alarm Rate

nu bounds the share of training points left outside the boundary. Check it on the training data. On new normal data the rate is usually somewhat higher, as some ordinary test points fall just outside.

In [ ]:
for nu in (0.01, 0.05, 0.1)
    m = svmtrain(Ztrain; svmtype = OneClassSVM, nu = nu)
    inside_train, _ = svmpredict(m, Ztrain)
    inside_test, _  = svmpredict(m, Ztest)
    println("nu = ", rpad(nu, 5), " training outside: ", round(100 * mean(.!inside_train), digits = 1), "%",
            "   test flagged: ", sum(.!inside_test))
end

## Summary
- The One-Class SVM learns a flexible boundary around clean training data.
- Standardise with the training statistics; LIBSVM expects observations in columns.
- nu sets the share of normal points flagged, the false-alarm rate.
- `svmpredict` returns `true` for points inside the boundary.